# Peer-group anomaly detection

This is the most exciting phase for me. You know those times where you know someone is fishy but they haven't broken any rules on paper, that is exactly the problem we are going to look in this section. I'm asking"is this customer's behavior statistically unusual compared to people like them, even if it doesn't match any named typology". Unlike the previous phases, it doesn't look for any rule breaks or triggers, it just looks and ask, "is there a significant deviation from peer norms".

This is a genuinely important layer in real AML systems, because typology-based rules only catch what you thought to look for. Statistical anomaly detection is the safety net for the unknown-unknowns

## Phase 4: Peer-Group Statistical Anomaly Detection

### Peer Group Definition

Peer groups are mainly based on **segment** (retail / SME / corporate) because these customer types have different transaction behaviours, as identified in Phase 1 and Phase 2. **Industry** is used as an additional level of grouping where the data is available.

In this dataset, `industry` is only available through the `employer_id` link in `employers.csv`, and this information exists only for retail customers. SME and corporate customers do not have an employer relationship, so there is no industry classification available for them. Because of this, retail customers can be grouped by both segment and industry, while SME and corporate customers are grouped by segment only. This is a limitation of the dataset rather than a design issue.

### Behavioral Dimensions

Three main behavioural measures are calculated for each customer at a monthly level:

* **Monthly volume** — the total value of all transactions, including both credits and debits, for each customer per month.

* **Velocity** — the total number of transactions made by each customer per month.

* **Counterparty entropy** — measures how concentrated or spread out a customer's transactions are across different counterparties. Low entropy means the customer mainly transacts with a small number of repeat counterparties, while high entropy means transactions are spread more evenly across many different counterparties. This adds information that transaction volume or count alone cannot capture. For example, a customer may have high transaction volume with one regular counterparty, which could be normal, while another customer may have the same volume spread across many different counterparties, which could be more unusual.

### Scoring Method: Log Z-Score, Not Percentile

Phase 1 showed that transaction amounts are heavily right-skewed and follow a roughly lognormal distribution. Because of this, using raw z-scores on transaction amounts could be distorted by very large outliers affecting the mean and standard deviation.

Instead of using percentile ranking like in Phase 2, this phase first applies a **log transform before z-scoring**:

`z = (log(x) - mean(log(x))) / std(log(x))`

The score is calculated within each peer group. This approach shows not only whether a customer is unusual, but also how far they are from normal behaviour within their group. For example, a customer with a score of 5 standard deviations is treated as more unusual than a customer with a score of 3. With percentile ranking, both very extreme customers may end up above roughly the 99th percentile, making it harder to distinguish how extreme each one actually is.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

DATA_DIR = '../data'

transactions = pd.read_csv(
    f'{DATA_DIR}/transactions.csv',
    parse_dates=['timestamp'],
    dtype={
        'transaction_id': 'str',
        'account_id': 'str',
        'customer_id': 'str',
        'transaction_type': 'category',
        'channel': 'category',
        'amount': 'float32',
        'currency': 'category',
        'amount_aed_equivalent': 'float32',
        'direction': 'category',
        'counterparty_id': 'str',
        'counterparty_name': 'str',
        'counterparty_country': 'category',
        'balance_after': 'float32'
    }
)

customers = pd.read_csv(f'{DATA_DIR}/customers.csv', parse_dates=['join_date'])
accounts = pd.read_csv(f'{DATA_DIR}/accounts.csv')
employers = pd.read_csv(f'{DATA_DIR}/employers.csv')
risk_profiles = pd.read_csv(f'{DATA_DIR}/customer_risk_profiles.csv')
ground_truth = pd.read_csv(f'{DATA_DIR}/ground_truth.csv')

print(f"Transactions: {transactions.shape}")
print(f"Customers: {customers.shape}")
print(f"Accounts: {accounts.shape}")
print(f"Employers: {employers.shape}")
print(f"Risk profiles: {risk_profiles.shape}")

Transactions: (5712869, 14)
Customers: (10000, 13)
Accounts: (11937, 4)
Employers: (1000, 3)
Risk profiles: (10000, 29)


I'm now going to join industry from employer table to retail customers

In [5]:
customers = customers.merge(
    employers[['employer_id', 'industry']],
    on='employer_id',
    how='left'
)

customers['peer_group'] = np.where(
    customers['segment'] == 'retail',
    'retail_' + customers['industry'].fillna('Unknown'),
    customers['segment']
)

customers['peer_group'].value_counts()

peer_group
sme                    1453
retail_Healthcare      1409
retail_Construction    1404
retail_Tech            1389
retail_Education       1389
retail_Retail          1354
retail_Finance         1137
corporate               465
Name: count, dtype: int64

 Calculating the first behavioral feature : Monthly volume and velocity per customer (Number of transactions). 

In [6]:
transactions['year_month'] = transactions['timestamp'].dt.to_period('M')

monthly_activity = transactions.groupby(['customer_id', 'year_month']).agg(
    monthly_volume=('amount_aed_equivalent', 'sum'),
    monthly_txn_count=('transaction_id', 'count')
).reset_index()

print(f"Customer-month rows: {len(monthly_activity)}")
monthly_activity.head()

Customer-month rows: 120000


,customer_id,year_month,monthly_volume,monthly_txn_count
0,CUST_100000,2023-01,10301.799805,8
1,CUST_100000,2023-02,8399.759766,11
2,CUST_100000,2023-03,10301.800781,6
3,CUST_100000,2023-04,7623.779785,11
4,CUST_100000,2023-05,7923.780273,11


In [7]:
monthly_activity = monthly_activity.merge(
    customers[['customer_id', 'segment', 'peer_group']],
    on='customer_id',
    how='left'
)

monthly_activity.head()

,customer_id,year_month,monthly_volume,monthly_txn_count,segment,peer_group
0,CUST_100000,2023-01,10301.799805,8,retail,retail_Finance
1,CUST_100000,2023-02,8399.759766,11,retail,retail_Finance
2,CUST_100000,2023-03,10301.800781,6,retail,retail_Finance
3,CUST_100000,2023-04,7623.779785,11,retail,retail_Finance
4,CUST_100000,2023-05,7923.780273,11,retail,retail_Finance


What this does, step by step: first sum each customer's transaction value per counterparty per month (so we know how much went to each individual counterparty), then apply the entropy formula across each customer-month's counterparty distribution. A customer transacting with only one counterparty in a month gets entropy = 0 (no dispersion at all); a customer with several counterparties getting roughly equal shares gets a higher entropy value.

In [8]:
def shannon_entropy(amounts):
    if len(amounts) <= 1:
        return 0.0
    total = amounts.sum()
    if total == 0:
        return 0.0
    shares = amounts / total
    shares = shares[shares > 0]
    return -np.sum(shares * np.log(shares))

cp_entropy = (
    transactions[transactions['counterparty_id'].notna()]
    .groupby(['customer_id', 'year_month', 'counterparty_id'])['amount_aed_equivalent']
    .sum()
    .reset_index()
    .groupby(['customer_id', 'year_month'])['amount_aed_equivalent']
    .apply(shannon_entropy)
    .rename('counterparty_entropy')
    .reset_index()
)

print(f"Customer-month entropy rows: {len(cp_entropy)}")
cp_entropy.head()

Customer-month entropy rows: 120000


,customer_id,year_month,counterparty_entropy
0,CUST_100000,2023-01,1.188943
1,CUST_100000,2023-02,1.027351
2,CUST_100000,2023-03,0.735635
3,CUST_100000,2023-04,0.798589
4,CUST_100000,2023-05,0.977183


### Counterparty Entropy: What It Actually Measures

In simple terms, counterparty entropy measures how spread out a customer's transactions are across different people or businesses each month. If someone regularly pays rent to the same landlord and shops at the same few places, most of their money goes to a small number of counterparties. This gives a low entropy score and is usually normal behaviour. If a customer's transactions are spread across many different counterparties, especially when the amounts are fairly evenly distributed, the entropy score becomes higher. (the natural log makes these extream values very small and entropy score thereby will be lower for large and small transactions).

This is useful for identifying layering type behaviour. One way money can be made harder to trace is by splitting it and moving it through many different accounts or entities instead of sending it directly to one destination. So, if a customer suddenly starts transacting with many new counterparties instead of their usual small group, this can be unusual even if none of the individual transactions are large enough to trigger an amount based rule.

The log transform used later when calculating z-scores has a different purpose. It reduces the impact of extremely large transaction amounts when comparing customers within a peer group. Without it, a few very large transactions could heavily affect the group average and make other unusual behaviour look normal by comparison. Applying the log transform reduces the effect of these extreme values while still keeping the differences between customers, giving a more stable and fair comparison of what is normal within the peer group.


In [9]:
monthly_activity = monthly_activity.merge(
    cp_entropy, on=['customer_id', 'year_month'], how='left'
)
monthly_activity['counterparty_entropy'] = monthly_activity['counterparty_entropy'].fillna(0)

monthly_activity[['monthly_volume', 'monthly_txn_count', 'counterparty_entropy']].describe()

,monthly_volume,monthly_txn_count,counterparty_entropy
count,1.200000e+05,120000.000000,120000.000000
mean,1.852734e+05,47.607242,1.108708
std,4.031467e+05,43.231849,0.415933
min,4.188810e+03,2.000000,0.066346
25%,3.038620e+04,20.000000,0.845309
50%,7.361266e+04,35.000000,1.101279
75%,1.811820e+05,57.000000,1.338100
max,1.077110e+07,308.000000,2.283018


### Apply log transform and compute z-scores within each peer group

In [12]:
monthly_activity['log_volume'] = np.log(monthly_activity['monthly_volume'])
monthly_activity['log_txn_count'] = np.log(monthly_activity['monthly_txn_count'])
monthly_activity['log_entropy'] = np.log(monthly_activity['counterparty_entropy'])

def zscore_within_group(df, col, group_col='peer_group'):
    return df.groupby(group_col)[col].transform(lambda x: (x - x.mean()) / x.std())

monthly_activity['z_volume'] = zscore_within_group(monthly_activity, 'log_volume')
monthly_activity['z_txn_count'] = zscore_within_group(monthly_activity, 'log_txn_count')
monthly_activity['z_entropy'] = zscore_within_group(monthly_activity, 'counterparty_entropy')

monthly_activity[['z_volume', 'z_txn_count', 'z_entropy']].describe()

,z_volume,z_txn_count,z_entropy
count,1.200000e+05,1.200000e+05,1.200000e+05
mean,-1.475016e-08,1.800042e-16,-2.644857e-08
std,9.999709e-01,9.999708e-01,9.999709e-01
min,-3.579065e+00,-4.867427e+00,-5.935954e+00
25%,-6.989254e-01,-5.947492e-01,-6.491759e-01
50%,-1.766749e-02,1.067487e-01,1.490257e-01
75%,6.672729e-01,7.091047e-01,7.393769e-01
max,4.500533e+00,2.303172e+00,2.474231e+00


For people don't know how standard deviation works. Standard deviation is simply a way of measuring how spread out the behaviour is within a group. You can think of it as showing the normal range based on the actual data rather than a limit chosen in advance. As a general rule, around 68% of values fall within 1 standard deviation of the average, and around 95% fall within 2 standard deviations. So if a customer's behaviour is within 2 standard deviations of their peer group's average, it would usually be considered normal variation. When the score moves further away, such as 3, 4 or more standard deviations, the behaviour becomes increasingly rare compared to the rest of the group and may be worth investigating.


### Combining these 3 z-scores into a composit score

In [13]:
monthly_activity['composite_anomaly_score'] = (
    monthly_activity[['z_volume', 'z_txn_count', 'z_entropy']]
    .abs() #using abs bcz the anomoly can be in both direction
    .mean(axis=1)
)

monthly_activity['composite_anomaly_score'].describe()

count    120000.000000
mean          0.805863
std           0.380989
min           0.010312
25%           0.519361
50%           0.753338
75%           1.052498
max           4.208789
Name: composite_anomaly_score, dtype: float64

In [14]:
monthly_activity.nlargest(15, 'composite_anomaly_score')[
    ['customer_id', 'year_month', 'peer_group', 'z_volume', 'z_txn_count', 'z_entropy', 'composite_anomaly_score']
]

,customer_id,year_month,peer_group,z_volume,z_txn_count,z_entropy,composite_anomaly_score
16170,CUST_101347,2023-07,corporate,-1.822987,-4.867427,-5.935954,4.208789
10629,CUST_100885,2023-10,sme,-2.774680,-4.121225,-5.211390,4.035765
104931,CUST_108744,2023-04,corporate,-3.310128,-4.867427,-3.088086,3.755214
40748,CUST_103395,2023-09,sme,-2.742391,-4.691885,-3.086715,3.506997
100796,CUST_108399,2023-09,corporate,-3.480219,-4.079774,-2.837415,3.465803
96069,CUST_108005,2023-10,sme,-2.930308,-4.482887,-2.939281,3.450825
68155,CUST_105679,2023-08,sme,-3.400301,-4.293825,-2.601944,3.432024
104934,CUST_108744,2023-07,corporate,-2.874493,-4.079774,-3.096646,3.350304
3703,CUST_100308,2023-08,corporate,-2.313787,-3.520924,-4.147762,3.327491
38134,CUST_103177,2023-11,sme,-2.824078,-4.293825,-2.840492,3.319465


These Z scores (being negative) shows that these customers had lower level of activity in terms of volume, transaction and entropy comppared to their peer. Also it is worth noting that the volume of these segmant of customers are quite low. 
    

In [15]:
print(monthly_activity['peer_group'].value_counts())
print()
print("Sign of top 200 anomalies' z_volume:")
print((monthly_activity.nlargest(200, 'composite_anomaly_score')['z_volume'] < 0).mean())

peer_group
sme                    17436
retail_Healthcare      16908
retail_Construction    16848
retail_Tech            16668
retail_Education       16668
retail_Retail          16248
retail_Finance         13644
corporate               5580
Name: count, dtype: int64

Sign of top 200 anomalies' z_volume:
0.645


In [16]:
top200 = monthly_activity.nlargest(200, 'composite_anomaly_score')
print(top200['peer_group'].value_counts())
print()
print("z_volume direction split in top 200:")
print((top200['z_volume'] > 0).value_counts())

peer_group
sme                    76
corporate              32
retail_Tech            30
retail_Healthcare      16
retail_Education       13
retail_Construction    12
retail_Finance         11
retail_Retail          10
Name: count, dtype: int64

z_volume direction split in top 200:
z_volume
False    129
True      71
Name: count, dtype: int64


### Finding: Smaller Peer Groups Produce More Extreme Z-Scores

The top 200 anomaly scored customer months are mostly coming from the `sme` group with 76 cases and the `corporate` group with 32 cases. Together, they make up 54% of the top anomalies even though these groups represent only around 19% of the total customer month population.

This does not necessarily mean SME and corporate customers are actually more unusual. Smaller peer groups, such as `corporate` with 465 customers and `sme` with 1,453 customers, can produce less stable estimates of the group average and standard deviation. This can result in more extreme z-scores, especially at the ends of the distribution, even when the underlying behaviour is not necessarily more unusual.

Another finding is that around two thirds of the top 200 anomalies are caused by unusually low activity, such as low transaction volume or a low number of transactions, rather than unusually high activity. These could represent dormant accounts or customers whose activity has recently reduced, which is different from the type of high activity usually associated with AML concerns.

I am documenting this as a limitation of using z-score based comparisons across peer groups with very different sizes rather than artificially changing the group sizes to improve the results. In a production system, this could be handled by setting a minimum peer group size before using z-scores or by using methods such as shrinkage or Bayesian approaches that make estimates from smaller groups more stable.


Now we want to define a cutof above which I will classify those customers as anomoly. Because our Z score is a composit, it is kind of difficult to apply 2 or 3 times of standard deviation from mean as the threshold. So we will have to find the threshold ourself

In [18]:
for pct in [90, 95, 97.5, 99, 99.5]:
    val = np.percentile(monthly_activity['composite_anomaly_score'], pct)
    print(f"{pct}th percentile: {val:.3f}")

90th percentile: 1.328
95th percentile: 1.487
97.5th percentile: 1.653
99th percentile: 1.827
99.5th percentile: 1.943


I'm going to use 97.5th percentile as the cutof, mainly because it allign with the practice of 2 standard deviation and it covers 97.5th transactions as Normal, so it will reduce false postive while capturing the most extream 2.5s.

In [19]:
ANOMALY_THRESHOLD = monthly_activity['composite_anomaly_score'].quantile(0.975)

monthly_activity['is_anomaly'] = monthly_activity['composite_anomaly_score'] >= ANOMALY_THRESHOLD

print(f"Threshold (97.5th percentile): {ANOMALY_THRESHOLD:.3f}")
print(f"Flagged customer-months: {monthly_activity['is_anomaly'].sum()}")
print(f"Unique customers flagged at least once: {monthly_activity[monthly_activity['is_anomaly']]['customer_id'].nunique()}")

Threshold (97.5th percentile): 1.653
Flagged customer-months: 3000
Unique customers flagged at least once: 1013


### validation against ground truth

In [20]:
flagged_customers = set(monthly_activity[monthly_activity['is_anomaly']]['customer_id'].unique())

scenario_customers_all = (
    ground_truth.merge(accounts[['account_id', 'customer_id']], on='account_id')
    ['customer_id'].unique()
)
scenario_customers_all = set(scenario_customers_all)

overlap = flagged_customers & scenario_customers_all
precision = len(overlap) / len(flagged_customers)
recall = len(overlap) / len(scenario_customers_all)

print(f"Total scenario-involved customers (ground truth, all types): {len(scenario_customers_all)}")
print(f"Total customers flagged by peer anomaly detection: {len(flagged_customers)}")
print(f"Overlap: {len(overlap)}")
print(f"Precision: {precision:.1%}")
print(f"Recall: {recall:.1%}")

Total scenario-involved customers (ground truth, all types): 580
Total customers flagged by peer anomaly detection: 1013
Overlap: 115
Precision: 11.4%
Recall: 19.8%


In [23]:
high_crr_customers = set(risk_profiles[risk_profiles['crr_tier'] == 'High']['customer_id'])

overlap_with_crr = flagged_customers & high_crr_customers
print(f"High CRR customers: {len(high_crr_customers)}")
print(f"Overlap between anomaly-flagged and High CRR: {len(overlap_with_crr)}")
print(f"% of anomaly-flagged customers who are also High CRR: {len(overlap_with_crr)/len(flagged_customers):.1%}")
print(f"% of High CRR customers also anomaly-flagged: {len(overlap_with_crr)/len(high_crr_customers):.1%}")

High CRR customers: 707
Overlap between anomaly-flagged and High CRR: 81
% of anomaly-flagged customers who are also High CRR: 8.0%
% of High CRR customers also anomaly-flagged: 11.5%


### Validation Results: Peer Anomaly Detection vs. Ground Truth and CRR

It is important to note what is actually being validated here. `ground_truth.csv` does not contain a separate label for general behavioural anomalies. It only identifies transactions linked to the four injected scenarios: Structuring, Rapid Movement, Layering and False Positive. Because of this, I checked whether customers flagged by peer anomaly detection also appeared in any of these known scenarios. This only gives an indication of whether the method identifies known scenario customers at all, rather than providing a true precision and recall test against a matching anomaly label like the typology specific rules in Phase 3. So this is a weaker form of validation and the results should be interpreted with that limitation in mind.

Peer group anomaly detection flagged 1,013 unique customers across 3,000 customer months using the 97.5th percentile threshold.

* **Against ground truth scenario customers:** The method achieved 11.4% precision and 19.8% recall, with 115 out of 580 customers involved in known scenarios also appearing among the 1,013 flagged customers.

* **Against Phase 2 High CRR (EDD) customers:** 81 out of 707 High CRR customers, or 11.5%, were also flagged as anomalies. At the same time, 8.0% of anomaly flagged customers were independently rated as High CRR.

These overlaps are modest but still higher than what would be expected from completely random selection, so they provide a small positive signal rather than strong validation. The result is clearly weaker than the typology specific detection rules in Phase 3, but I would not treat that as a failure because this method is designed to detect a different type of behaviour. Instead of looking for specific patterns such as Structuring or Rapid Movement, it identifies customers whose volume, transaction activity or counterparty behaviour is unusual compared to similar customers.

The earlier analysis also showed that the current anomaly score is more likely to identify unusually low activity, such as dormant or suddenly quiet accounts, than the high value transaction bursts seen in the injected AML scenarios. In a real system, the main value of this approach would be finding new or unusual behaviour that existing rules were not specifically designed to detect. Because of this, its performance cannot be fully measured by how much it overlaps with typologies that already have their own dedicated detection rules elsewhere in the system.


## Phase 4 Summary: Peer-Group Statistical Anomaly Detection

Built a behavioural anomaly detection score that compares each customer’s monthly activity against similar customers in their peer group. Customer segment was used as the main grouping, while industry was used as an additional grouping for retail customers where the information was available. SME and corporate customers were grouped by segment only because industry information was not available for them in this dataset.

The score combines three behavioural measures: monthly transaction volume, transaction count or velocity, and counterparty entropy, which measures how spread out transactions are across different counterparties. Volume and transaction count were log transformed before calculating z-scores because their distributions were heavily right skewed, as identified in Phase 1. Counterparty entropy was z-scored directly because it is already a bounded metric. I initially applied a log transform to entropy as well, but this caused distortion at the lower end of the distribution and was corrected during development.

The 97.5th percentile of the final composite anomaly score was used as the threshold for identifying unusual customer months. This flagged 1,013 unique customers across 3,000 customer months.

**Key findings:**

* Smaller peer groups, particularly SME and corporate customers, produced a disproportionately high number of extreme z-scores because smaller groups give less stable estimates of the mean and standard deviation. This is a statistical limitation and does not necessarily mean these customer groups are genuinely more unusual.

* The composite score was more likely to flag unusually low activity, such as dormant or suddenly quiet accounts, rather than unusually high activity. This is a different type of behaviour from the high value and deliberately generated AML typologies in the ground truth data.

* Validation against ground truth showed 11.4% precision and 19.8% recall, while 11.5% of Phase 2 High CRR customers also overlapped with anomaly flagged customers. These results show a modest but real relationship, although they are weaker than the typology specific rules in Phase 3.

* The validation should also be interpreted carefully because the ground truth contains no separate label for general behavioural anomalies. The comparison only checks whether anomaly flagged customers overlap with customers involved in the known injected scenarios, making it a proxy check rather than a direct precision and recall test.

Overall, I see this layer as a complementary safety net alongside the Customer Risk Rating model in Phase 2 and the typology specific detection rules in Phase 3. It is designed to identify unusual behaviour that the existing rules were not specifically built to detect, rather than replace those approaches.


In [24]:
monthly_activity.to_csv('../data/peer_anomaly_scores.csv', index=False)
monthly_activity.shape

(120000, 15)